[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/2_Practica/reto_expres/Reto_expres_texto.ipynb)

# Reto exprés · Texto: diseña, entrena y defiende tu red

Un único cuaderno para los 3 problemas de texto. **Solo se tocan las celdas ✏️**. Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · elegir problema (3 valores)
- ✏️ **2** · comparar tamaños de red **y el modelo simple**, y elegir **mirando solo la validación**
- ✏️ **3** · prueba una sola vez y dónde se equivoca
- ✏️ **4** · probar la aplicación con una frase vuestra
- Al final sale una **FICHA** con todos los números para vuestra presentación.

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, urllib.request, time
BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"
def dato(nombre):
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta): return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tensorflow import keras
SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
print("Listo ✔")

### ✏️ 1 · Elegid el problema

| ARCHIVO | COLUMNAS_TEXTO | OBJETIVO |
|---|---|---|
| `resenas.csv` | `["texto"]` | `sentimiento` (positivo, negativo, neutro) |
| `mensajes_soporte.csv` | `["texto"]` | `categoria` (factura, avería, contratar, baja) |
| `correos.csv` | `["asunto", "texto"]` | `es_spam` (0 / 1) |

In [ ]:
ARCHIVO = "resenas.csv"
COLUMNAS_TEXTO = ["texto"]
OBJETIVO = "sentimiento"

### ⚙️ Preparar los textos — pulsa ▶

Tres partes: **entrenamiento** (aprende), **validación** (para elegir) y **prueba** (se mira una sola vez). El texto se convierte en números con **TF-IDF** (qué palabras pesan en cada mensaje).

In [ ]:
df = pd.read_csv(dato(ARCHIVO)).dropna(subset=COLUMNAS_TEXTO + [OBJETIVO])
textos = df[COLUMNAS_TEXTO].astype(str).agg(" ".join, axis=1).to_numpy()
if OBJETIVO == "es_spam": df[OBJETIVO] = df[OBJETIVO].map({0: "no_spam", 1: "spam"})
codigos, NOMBRES = pd.factorize(df[OBJETIVO].astype(str)); NOMBRES = [str(n) for n in NOMBRES]
y = codigos; N_CLASES = len(NOMBRES)
T_res, T_pru, y_res, y_pru = train_test_split(textos, y, test_size=0.2, random_state=SEMILLA, stratify=y)
T_ent, T_val, y_ent, y_val = train_test_split(T_res, y_res, test_size=0.25, random_state=SEMILLA, stratify=y_res)
tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=3000)
A_ent = tfidf.fit_transform(T_ent).toarray().astype("float32")
A_val = tfidf.transform(T_val).toarray().astype("float32"); A_pru = tfidf.transform(T_pru).toarray().astype("float32")
LINEA_BASE = np.bincount(y_pru).max() / len(y_pru)
print(f"{len(df)} textos · {A_ent.shape[1]} palabras/pares en el vocabulario · clases: {NOMBRES}")
print(f"Ejemplo: «{T_ent[0][:110]}» → {NOMBRES[y_ent[0]]}")
print(f"LÍNEA BASE: diciendo siempre la clase más habitual se acierta el {LINEA_BASE:.1%}.")

### ⚙️ Las dos candidatas — pulsa ▶

- **modelo simple**: regresión logística sobre TF-IDF. Es el rival a batir.
- **red**: una capa oculta de N neuronas sobre TF-IDF.

In [ ]:
def construir(neuronas):
    m = keras.Sequential([keras.layers.Input(shape=(A_ent.shape[1],)), keras.layers.Dropout(0.2),
                          keras.layers.Dense(neuronas, activation="relu"), keras.layers.Dense(N_CLASES, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def entrenar(neuronas, epocas=40):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(neuronas)
    parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    h = m.fit(A_ent, y_ent, validation_data=(A_val, y_val), epochs=epocas, batch_size=32, callbacks=[parar], verbose=0)
    return m, h

simple = LogisticRegression(max_iter=1000).fit(A_ent, y_ent)
print("Listo ✔")

### ✏️ 2 · Probar tamaños y elegir (solo con la validación)

Cambiad **PRUEBAS** (mínimo 3 valores). Mirad **val_acc** y **val_f1** (F1 medio de todas las clases: más alto = mejor) y comparadlo con la fila del **modelo simple**. Escribid en **ELEGIDA** lo que justificáis: puede ser una red… o quedarse con el modelo simple si gana.

In [ ]:
PRUEBAS = [4, 16, 64]      # neuronas de la capa oculta
pv = simple.predict(A_val)
filas = [dict(candidata="modelo simple (regresión logística)", val_acc=round(accuracy_score(y_val, pv), 3),
              val_f1=round(f1_score(y_val, pv, average="macro"), 3), epocas="-", segundos="-")]
for n in PRUEBAS:
    t0 = time.time(); m, h = entrenar(n); pr = m.predict(A_val, verbose=0).argmax(1)
    filas.append(dict(candidata=f"red de {n} neuronas", val_acc=round(accuracy_score(y_val, pr), 3),
                      val_f1=round(f1_score(y_val, pr, average="macro"), 3), epocas=len(h.history["loss"]), segundos=round(time.time() - t0)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = 16               # ✏️ neuronas elegidas (o poned 0 si os quedáis con el modelo simple)

### ⚙️ Entrenar la elegida y ver las curvas — pulsa ▶

Si la de validación sube mientras la de entrenamiento baja, hay sobreajuste (con textos pequeños pasa enseguida).

In [ ]:
if ELEGIDA > 0:
    modelo, hist = entrenar(ELEGIDA)
    plt.plot(hist.history["loss"], label="entrenamiento"); plt.plot(hist.history["val_loss"], label="validación")
    plt.xlabel("época"); plt.ylabel("pérdida"); plt.title(f"{ELEGIDA} neuronas"); plt.legend(); plt.show()
    predecir = lambda A: modelo.predict(A, verbose=0)
else:
    predecir = lambda A: simple.predict_proba(A)
    print("Os quedáis con el modelo simple: no hay curvas que ver.")

### ✏️ 3 · La prueba, una sola vez, y dónde se equivoca

- **F1 por clase**: mezcla precisión y recall. **La peor clase** cuenta más que la media.
- Leed los **fallos**: ¿son frases ambiguas (que también confundiría una persona) o la red no ha entendido nada?
- Comparad con el modelo simple **en la prueba**: ¿le gana de verdad la red?

In [ ]:
p = predecir(A_pru); c = p.argmax(1)
acc = accuracy_score(y_pru, c); f1 = f1_score(y_pru, c, average="macro")
acc_s = accuracy_score(y_pru, simple.predict(A_pru)); f1_s = f1_score(y_pru, simple.predict(A_pru), average="macro")
inf = pd.DataFrame(classification_report(y_pru, c, target_names=NOMBRES, output_dict=True, zero_division=0)).T.iloc[:N_CLASES]
display(inf[["precision", "recall", "f1-score", "support"]].round(2))
peor = inf["f1-score"].idxmin()
cm = confusion_matrix(y_pru, c); fuera = cm - np.diag(np.diag(cm)); r, k = np.unravel_index(fuera.argmax(), fuera.shape)
print("Matriz de confusión (filas = realidad, columnas = lo que dijo):"); display(pd.DataFrame(cm, index=NOMBRES, columns=NOMBRES))
print("Algunos fallos:")
for i in np.where(c != y_pru)[0][:6]:
    print(f"  «{T_pru[i][:100]}»  → real: {NOMBRES[y_pru[i]]} · dijo: {NOMBRES[c[i]]}")
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ clasificar «{OBJETIVO}» ({ARCHIVO}) · {len(df)} textos · {N_CLASES} clases
Candidatas ...... {[(f['candidata'], f['val_f1']) for f in filas]}  (val_f1)
Elegida ......... {"red de %d neuronas" % ELEGIDA if ELEGIDA > 0 else "modelo simple"}
--- PRUEBA (una sola vez) ---
Línea base ...... {LINEA_BASE:.1%}   (acertar sin modelo)
Exactitud ....... {acc:.1%}   · F1 medio {f1:.3f}
Modelo simple ... {acc_s:.1%}   · F1 medio {f1_s:.3f}   {"(la red gana)" if f1 > f1_s + 0.01 else "(la red NO gana de forma clara)"}
Peor clase ...... {peor} (F1 {inf.loc[peor, "f1-score"]:.2f})
Confusión mayor . {NOMBRES[r]} → {NOMBRES[k]}  ({fuera[r, k]} casos)
============================================================
""")

### ✏️ 4 · La aplicación: probar una frase vuestra

Escribid cualquier frase dentro de `probar("...")`. Probad frases ambiguas, con faltas o con ironía.

In [ ]:
def probar(frase):
    q = predecir(tfidf.transform([frase]).toarray().astype("float32"))[0]; top = q.argsort()[::-1]
    print(f"«{frase}»\n  → {NOMBRES[top[0]]} ({q[top[0]]:.0%})   otras: " + ", ".join(f"{NOMBRES[t]} {q[t]:.0%}" for t in top[1:3]))

probar(T_pru[0]); probar("escribe aquí tu propia frase")

### 🎤 Para la presentación

Copiad la **FICHA**, la tabla de la celda ✏️ 2 y la matriz de confusión. Una frase para cada pregunta: **¿gana al modelo simple? · ¿por qué ese tamaño? · ¿qué frases confunde? · ¿lo usaríamos de verdad?**